1. ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ

In [ ]:
'''Ячейка номер: 1.1. Назначение: Импортируем зависимости которые понадобятся для дальнейшей работы'''

import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
#from pycaret.clustering import *
#from sklearn.datasets import make_blobs
#mpl.rcParams['figure.dpi'] = 100 #300 сделает картинки большими

from IPython.display import display, clear_output
from ipywidgets import Dropdown
import ipywidgets as widgets
from ipywidgets import IntSlider
import io


import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns', None)

 # Глобальная переменная
#data_tmp = None 
#list_of_feature = None
#city_cut = None

In [21]:
'''Ячейка номер: 1.2. Назначение: Импортируем и преобразуем в нужные форматы данные по On-Shelf Availability (OSA). Это параметр который представляет собой соотношение кол-ва продуктов которые фактически были на полке 
 полках в торговой точке в момент визита, к количеству продуктов которое потенциально могло там быть. '''

osa_path = r'C:\Users\rokotyev\Yandex.Disk\_Main Data Rokotyan\2. Проекты\60. Подбор пар ТТ для тестов\_data\osa'
df_osa = pd.DataFrame()
for filename in os.listdir(osa_path):
    df_tmp = pd.read_csv(osa_path + '\\' + filename, sep=';', dtype='str')
    df_osa = pd.concat([df_osa, df_tmp])
    df_tmp = pd.DataFrame()


df_osa['VISIT_DATE'] = pd.to_datetime(df_osa['VISIT_DATE'])
df_osa[['SHIP_TO']] = df_osa[['SHIP_TO']].astype('int')
df_osa[['OSA']] = df_osa[['OSA']].astype('float')
#
df_osa['YEAR'] = df_osa['VISIT_DATE'].dt.year
df_osa['MONTH'] = df_osa['VISIT_DATE'].dt.month
df_osa['WEEK'] = df_osa['VISIT_DATE'].dt.isocalendar().week
df_osa[['YEAR', 'MONTH','WEEK']] = df_osa[['YEAR', 'MONTH','WEEK']].astype('int')
df_osa['YEAR_MONTH'] = ''
df_osa['YEAR_WEEK'] = ''
df_osa.loc[df_osa['MONTH'] < 10, 'YEAR_MONTH'] = '0'
df_osa.loc[df_osa['WEEK'] < 10, 'YEAR_WEEK'] = '0'
df_osa['YEAR_MONTH'] = df_osa['YEAR'].astype('str') + df_osa['YEAR_MONTH'] + df_osa['MONTH'].astype('str') 
df_osa['YEAR_MONTH'] = df_osa['YEAR_MONTH'].astype('int')

df_osa['YEAR_WEEK'] = df_osa['YEAR'].astype('str') + df_osa['YEAR_WEEK'] + df_osa['WEEK'].astype('str') 
df_osa['YEAR_WEEK'] = df_osa['YEAR_WEEK'].astype('int')

#df_osa.head(2)

In [ ]:

''''''

# http://10.179.161.142:8181/direct?id=38f6fa69-c45d-4074-a588-3c0e25693f67

sales_path = r'C:\Users\rokotyev\Yandex.Disk\_Main Data Rokotyan\2. Проекты\60. Подбор пар ТТ для тестов\_data\sales'
df_sales = pd.DataFrame()
for filename in os.listdir(sales_path):
    df_tmp = pd.read_csv(sales_path + '\\' + filename, sep=',', dtype='str')
    df_sales = pd.concat([df_sales, df_tmp])
    df_tmp = pd.DataFrame()
    
df_sales.rename(columns={'(CUS) Ship To Code': 'SHIP_TO', '(CUS-DC) Customer Group':'CUSTOMER_GROUP', '(CUS-DC) Proxi Category 2':'CLIENT', '(CUS-DC) Proxi Category 3':'CHANNEL', 'Week of Year (ID)':'YEAR_WEEK'}, inplace=True)
df_sales = df_sales.query("CHANNEL in ('NATIONAL KEY ACCOUNT', 'LOCAL KEY ACCOUNT')")
df_sales.drop(columns=['(CUS) Ship To Address', '(CUS) Ship To', 'Client MacroGroup' ], inplace=True)
#df_sales['VISIT_DATE'] = pd.to_datetime(df_sales['VISIT_DATE'])
df_sales[['SHIP_TO', 'YEAR_WEEK']] = df_sales[['SHIP_TO', 'YEAR_WEEK']].astype('int')
df_sales['kCAF'] = df_sales['kCAF'].astype('float')


#df_sales.head(10)

In [ ]:
visits_path = r'C:\Users\rokotyev\Yandex.Disk\_Main Data Rokotyan\2. Проекты\60. Подбор пар ТТ для тестов\_data\visits'
df_visits = pd.DataFrame()
for filename in os.listdir(visits_path):
    df_tmp = pd.read_csv(visits_path + '\\' + filename, sep=';', dtype='str')
    df_visits = pd.concat([df_visits, df_tmp])
    df_tmp = pd.DataFrame()
    
df_visits['VISIT_DATE'] = pd.to_datetime(df_visits['VISIT_DATE'])
df_visits['SHIP_TO'] = df_visits['SHIP_TO'].astype('int')

df_visits['YEAR'] = df_visits['VISIT_DATE'].dt.year
df_visits['MONTH'] = df_visits['VISIT_DATE'].dt.month
df_visits['WEEK'] = df_visits['VISIT_DATE'].dt.isocalendar().week
df_visits[['YEAR', 'MONTH', 'WEEK']] = df_visits[['YEAR', 'MONTH','WEEK']].astype('int')
df_visits['YEAR_MONTH'] = ''
df_visits['YEAR_WEEK'] = ''
df_visits.loc[df_visits['MONTH'] < 10, 'YEAR_MONTH'] = '0'
df_visits.loc[df_visits['WEEK'] < 10, 'YEAR_WEEK'] = '0'
df_visits['YEAR_MONTH'] = df_visits['YEAR'].astype('str') + df_visits['YEAR_MONTH'] + df_visits['MONTH'].astype('str') 
df_visits['YEAR_MONTH'] = df_visits['YEAR_MONTH'].astype('int')
df_visits['YEAR_WEEK'] = df_visits['YEAR'].astype('str') + df_visits['YEAR_WEEK'] + df_visits['WEEK'].astype('str') 
df_visits['YEAR_WEEK'] = df_visits['YEAR_WEEK'].astype('int')

## Блок аггрегации

df_visits = df_visits.groupby(by=['YEAR_WEEK', 'SHIP_TO']).agg({'PHOTO_AUDIT_ID':'size'}).reset_index()
df_visits.rename(columns={'PHOTO_AUDIT_ID': 'PHOTO_AUDITS_PER_WEEK'}, inplace=True)


#df_visits.head(2)

In [ ]:
picos_path = r'C:\Users\rokotyev\Yandex.Disk\_Main Data Rokotyan\2. Проекты\60. Подбор пар ТТ для тестов\_data\picos'
df_picos = pd.DataFrame()
for filename in os.listdir(picos_path):
    df_tmp = pd.read_csv(picos_path + '\\' + filename, sep=';', dtype='str')
    df_picos = pd.concat([df_picos, df_tmp])
    df_tmp = pd.DataFrame()
    
df_picos['VISIT_DATE'] = pd.to_datetime(df_picos['VISIT_DATE'])
df_picos['SHIP_TO'] = df_picos['SHIP_TO'].astype('int')
df_picos['PICOS_SCORE_FACT'] = df_picos['PICOS_SCORE_FACT'].astype('float').astype('int')
df_picos['YEAR'] = df_picos['VISIT_DATE'].dt.year
df_picos['MONTH'] = df_picos['VISIT_DATE'].dt.month
df_picos['WEEK'] = df_picos['VISIT_DATE'].dt.isocalendar().week
df_picos[['YEAR', 'MONTH', 'WEEK']] = df_picos[['YEAR', 'MONTH','WEEK']].astype('int')
df_picos['YEAR_MONTH'] = ''
df_picos['YEAR_WEEK'] = ''
df_picos.loc[df_picos['MONTH'] < 10, 'YEAR_MONTH'] = '0'
df_picos.loc[df_picos['WEEK'] < 10, 'YEAR_WEEK'] = '0'
df_picos['YEAR_MONTH'] = df_picos['YEAR'].astype('str') + df_picos['YEAR_MONTH'] + df_picos['MONTH'].astype('str') 
df_picos['YEAR_MONTH'] = df_picos['YEAR_MONTH'].astype('int')
df_picos['YEAR_WEEK'] = df_picos['YEAR'].astype('str') + df_picos['YEAR_WEEK'] + df_picos['WEEK'].astype('str') 
df_picos['YEAR_WEEK'] = df_picos['YEAR_WEEK'].astype('int')

#df_picos.head(2)

In [ ]:
facing_path = r'C:\Users\rokotyev\Yandex.Disk\_Main Data Rokotyan\2. Проекты\60. Подбор пар ТТ для тестов\_data\facing'
df_facing_fact = pd.DataFrame()
for filename in os.listdir(facing_path):
    df_tmp = pd.read_csv(facing_path + '\\' + filename, sep=';', dtype='str')
    df_facing_fact = pd.concat([df_facing_fact, df_tmp])
    df_tmp = pd.DataFrame()
    
df_facing_fact['VISIT_DATE'] = pd.to_datetime(df_facing_fact['VISIT_DATE'])
df_facing_fact['SHIP_TO'] = df_facing_fact['SHIP_TO'].astype('int')
df_facing_fact['GROUP_FACT'] = df_facing_fact['GROUP_FACT'].astype('float').astype('int')
df_facing_fact['YEAR'] = df_facing_fact['VISIT_DATE'].dt.year
df_facing_fact['MONTH'] = df_facing_fact['VISIT_DATE'].dt.month
df_facing_fact['WEEK'] = df_facing_fact['VISIT_DATE'].dt.isocalendar().week
df_facing_fact[['YEAR', 'MONTH', 'WEEK']] = df_facing_fact[['YEAR', 'MONTH','WEEK']].astype('int')
df_facing_fact['YEAR_MONTH'] = ''
df_facing_fact['YEAR_WEEK'] = ''
df_facing_fact.loc[df_facing_fact['MONTH'] < 10, 'YEAR_MONTH'] = '0'
df_facing_fact.loc[df_facing_fact['WEEK'] < 10, 'YEAR_WEEK'] = '0'
df_facing_fact['YEAR_MONTH'] = df_facing_fact['YEAR'].astype('str') + df_facing_fact['YEAR_MONTH'] + df_facing_fact['MONTH'].astype('str') 
df_facing_fact['YEAR_MONTH'] = df_facing_fact['YEAR_MONTH'].astype('int')
df_facing_fact['YEAR_WEEK'] = df_facing_fact['YEAR'].astype('str') + df_facing_fact['YEAR_WEEK'] + df_facing_fact['WEEK'].astype('str') 
df_facing_fact['YEAR_WEEK'] = df_facing_fact['YEAR_WEEK'].astype('int')

#df_facing_fact.head(2)

In [ ]:
df_pos_list = pd.read_csv(r"C:\Users\rokotyev\Yandex.Disk\_Main Data Rokotyan\2. Проекты\60. Подбор пар ТТ для тестов\_data\dim_poslist.csv", sep=';', dtype='str')
df_pos_list[['SHIP_TO', 'fid']] = df_pos_list[['SHIP_TO', 'fid']].astype('int')
df_pos_list[['LATITUDE', 'LONGITUDE']] = df_pos_list[['LATITUDE', 'LONGITUDE']].astype('float')

#print(df_pos_list.shape)
#df_pos_list.dtypes

2. ЗАГРУЗКА СПИСКА ТЕСТОВЫХ ТОЧЕК

In [ ]:


test_pos_list = []

# Создаём виджет загрузки (только .xlsx файлы)
uploader = widgets.FileUpload(accept='.xlsx',
                              multiple=False,
                              description='Загрузка списка тестовых ТТ',
                              layout=widgets.Layout(width='20%', height='40px'),
                              button_style='warning'                              
                              )
output_file = widgets.Output()

def on_upload(change):
    global test_pos_list
    if uploader.value:
        # Получаем первый загруженный файл
        uploaded_file = uploader.value[0]
        content = uploaded_file['content']
        print(content)
        # Читаем Excel из байтов
        test_pos_list = pd.read_excel(io.BytesIO(content))
        with output_file:
            clear_output()
            print("Загружено строк: " + str(test_pos_list.shape[0]) )
            #display(test_pos_list.head(5))  # Показываем первые строки

uploader.observe(on_upload, names='value')
display(widgets.VBox([uploader, output_file]))

3. ВЫБОР ПАРАМЕТРОВ ПОДБОРА ПАР

In [ ]:
from IPython.display import display
import ipywidgets as widgets

# Удобный выбор территориальных признаков
feature_list_geo = []
options_geo = ['RSD', 'BUSINESS_UNIT', 'SALES_GROUP', 'CLIENT', 'CUSTOMER_GROUP', 'CITY']

# Создаем чекбоксы
checkboxes_geo = [widgets.Checkbox(value=False, description=option_geo) for option_geo in options_geo]

# Блок вывода
output_feature = widgets.Output()

# Функция обновления списка выбранных фич
def on_change_feature(change):
    global feature_list_geo
    selected_geo = sorted([
        checkbox_geo.description
        for checkbox_geo in checkboxes_geo
        if checkbox_geo.value
    ])
    feature_list_geo = selected_geo

    with output_feature:
        output_feature.clear_output()
        print(f'Для поиска пар будут использоваться только точки с одинаковыми: {feature_list_geo}')

# Подписываем каждый чекбокс на изменение
for checkbox_geo in checkboxes_geo:
    checkbox_geo.observe(on_change_feature, names='value')

# Отображаем элементы
wig = widgets.VBox([
    widgets.GridBox(
        checkboxes_geo,
        layout=widgets.Layout(grid_template_columns="repeat(2, 250px)")
    )
])

display(wig, output_feature)

In [ ]:
# Удобный выбор временного периода

options_week =  sorted(list(map(str, df_sales['YEAR_WEEK'].unique())))
week_list = options_week.copy()

# Создаем чекбоксы для каждой опции
checkboxes_week = [widgets.Checkbox(value=True, description=option) for option in options_week]


# Функция для обработки нажатия на отдельный чек-бокс - записываем список выбранных фич в глоб. переменную feature_list
def on_change_week(change):
    global week_list
    selected = sorted([checkbox.description for checkbox in checkboxes_week if checkbox.value])
    week_list = selected  # Объявляем, что используем глобальную переменную
    with output_week: # Этот блок печатает список
         output_week.clear_output()  
         print(f'Для поиска пар будут расчитаны KPI только за выбранный период: {week_list}')  


# Запускаем мониторинг изменений на каждом чекбоксе (событие observe означает изменеие )
for checkbox in checkboxes_week:
    checkbox.observe(on_change_week, names='value')

# Отображаем чекбоксы на экране
output_week = widgets.Output()



wig_week = widgets.VBox([widgets.GridBox(checkboxes_week, layout=widgets.Layout(grid_template_columns="repeat(5, 250px)"))])
display(wig_week, output_week)


#### ВЫБОР KPI ДЛЯ ПОДБОРА ПАР:

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

options_kpi = ['kCAF', 'OSA', 'VISITS', 'PICOS', 'FACING']

kpi_list = []
kpi_ranges = {}
checkboxes = {}
sliders = {}
rows = {}

output = widgets.Output()

def sync_globals():
    global kpi_list, kpi_ranges
    kpi_list = [name for name in options_kpi if checkboxes[name].value]
    kpi_ranges = {name: sliders[name].value for name in kpi_list}
    for name in kpi_list:
        globals()[f'kpi_{name}'] = sliders[name].value

def redraw_output():
    with output:
        clear_output(wait=True)
        print(f"Выбранные KPI: {kpi_list}")
        #print("Диапазоны KPI:")
        #for name in kpi_list:
        #    print(f"{name}: {sliders[name].value}")

def on_checkbox_change(change):
    kpi_name = change.owner.description
    if change.new:
        sliders[kpi_name].layout.display = 'flex'
    else:
        sliders[kpi_name].layout.display = 'none'
        if f'kpi_{kpi_name}' in globals():
            globals().pop(f'kpi_{kpi_name}', None)
    sync_globals()
    redraw_output()

def on_slider_change(change):
    kpi_name = change.owner.description
    globals()[f'kpi_{kpi_name}'] = change.new
    sync_globals()
    redraw_output()

for name in options_kpi:
    cb = widgets.Checkbox(value=False, description=name, layout=widgets.Layout(width='150px'))
    sl = widgets.FloatRangeSlider(
        value=(0.5, 1.5),
        min=0.0,
        max=3.0,
        step=0.1,
        description='Откл. %',
        continuous_update=False,
        layout=widgets.Layout(width='350px', display='none')
    )

    cb.observe(on_checkbox_change, names='value')
    sl.observe(on_slider_change, names='value')

    checkboxes[name] = cb
    sliders[name] = sl
    rows[name] = widgets.HBox([cb, sl])

ui = widgets.VBox([rows[name] for name in options_kpi])

display(ui, output)

sync_globals()
redraw_output()

3. БЛОК ПОИСКА ПАР

In [ ]:
#Обрезамем данные по KPI согласно выбранным месяцам и считаем среднее по оставшимся

def cut_dataframes(week_list, df_sales, df_osa, df_visits, df_facing_fact, df_picos):
    se_week_list = pd.Series(map(int, week_list), name = 'YEAR_WEEK')

    df_sales_cut = pd.DataFrame(df_sales)
    df_sales_cut = df_sales.merge(se_week_list, on='YEAR_WEEK', how='inner')
    df_sales_cut = df_sales_cut.groupby(by=['SHIP_TO']).agg({'kCAF':'mean' }).reset_index()
    df_sales_cut['kCAF'] = df_sales_cut['kCAF'].astype('int')

    df_osa_cut = pd.DataFrame(df_osa)
    df_osa_cut = df_osa_cut.merge(se_week_list, on='YEAR_WEEK', how='inner')
    df_osa_cut = df_osa_cut.groupby(by=['SHIP_TO']).agg({'OSA':'mean' }).reset_index()
    df_osa_cut['OSA'] = df_osa_cut['OSA'].astype('float')

    df_visits_cut = pd.DataFrame(df_visits)
    df_visits_cut = df_visits_cut.merge(se_week_list, on='YEAR_WEEK', how='inner')
    df_visits_cut = df_visits_cut.groupby(by=['SHIP_TO']).agg({'PHOTO_AUDITS_PER_WEEK':'mean' }).reset_index()
    df_visits_cut['PHOTO_AUDITS_PER_WEEK'] = df_visits_cut['PHOTO_AUDITS_PER_WEEK'].astype('int')

    df_facing_fact_cut = pd.DataFrame(df_facing_fact)
    df_facing_fact_cut = df_facing_fact_cut.merge(se_week_list, on='YEAR_WEEK', how='inner')
    df_facing_fact_cut = df_facing_fact_cut.groupby(by=['SHIP_TO']).agg({'GROUP_FACT':'mean' }).reset_index()
    df_facing_fact_cut['GROUP_FACT'] = df_facing_fact_cut['GROUP_FACT'].astype('int')

    df_picos_cut = pd.DataFrame(df_picos)
    df_picos_cut = df_picos_cut.merge(se_week_list, on='YEAR_WEEK', how='inner')
    df_picos_cut = df_picos_cut.groupby(by=['SHIP_TO']).agg({'PICOS_SCORE_FACT':'mean' }).reset_index()
    df_picos_cut['PICOS_SCORE_FACT'] = df_picos_cut['PICOS_SCORE_FACT'].astype('int')

    return df_sales_cut, df_osa_cut, df_visits_cut, df_facing_fact_cut, df_picos_cut

In [ ]:
# Для точек плдтягиваем KPI

def calc_kpi_for_test_pos(df_pos_list, df_sales_cut, df_osa_cut, df_visits_cut, df_facing_fact_cut, df_picos_cut ):

    df_pos_list_with_kpi = pd.DataFrame(df_pos_list)

    if 'OSA' in kpi_list: 
        df_pos_list_with_kpi = df_pos_list_with_kpi.merge(df_osa_cut, on='SHIP_TO', how='left', indicator=True )
        #df_pos_list = df_pos_list.loc[df_pos_list['_merge'] != 'right_only'].reset_index().drop(columns='index')
        df_pos_list_with_kpi['_merge'] = df_pos_list_with_kpi['_merge'].astype('string')
        df_pos_list_with_kpi.rename(columns={'_merge': 'DATA_IN_OSA' }, inplace=True)
        df_pos_list_with_kpi.replace({'DATA_IN_OSA' : { 'both' : 'Ok', 'left_only' : 'No_OSA_data'}}, inplace=True)

    if 'kCAF' in kpi_list: 
        df_pos_list_with_kpi = df_pos_list_with_kpi.merge(df_sales_cut, on='SHIP_TO', how='left', indicator=True )
        #df_pos_list = df_pos_list.loc[df_pos_list['_merge'] != 'right_only'].reset_index().drop(columns='index')
        df_pos_list_with_kpi['_merge'] = df_pos_list_with_kpi['_merge'].astype('string')
        df_pos_list_with_kpi.rename(columns={'_merge': 'DATA_IN_SALES' }, inplace=True)
        df_pos_list_with_kpi.replace({'DATA_IN_SALES' : { 'both' : 'Ok', 'left_only' : 'No_SALES_data'}}, inplace=True)

    if 'VISITS' in kpi_list: 
        df_pos_list_with_kpi = df_pos_list_with_kpi.merge(df_visits_cut, on='SHIP_TO', how='left', indicator=True )
        #df_pos_list = df_pos_list.loc[df_pos_list['_merge'] != 'right_only'].reset_index().drop(columns='index')
        df_pos_list_with_kpi['_merge'] = df_pos_list_with_kpi['_merge'].astype('string')
        df_pos_list_with_kpi.rename(columns={'_merge': 'DATA_IN_VISITS' }, inplace=True)
        df_pos_list_with_kpi.replace({'DATA_IN_VISITS' : { 'both' : 'Ok', 'left_only' : 'No_VISITS_data'}}, inplace=True)
    
    if 'FACING' in kpi_list: 
        df_pos_list_with_kpi = df_pos_list_with_kpi.merge(df_facing_fact_cut, on='SHIP_TO', how='left', indicator=True )
        #df_pos_list = df_pos_list.loc[df_pos_list['_merge'] != 'right_only'].reset_index().drop(columns='index')
        df_pos_list_with_kpi['_merge'] = df_pos_list_with_kpi['_merge'].astype('string')
        df_pos_list_with_kpi.rename(columns={'_merge': 'DATA_IN_FACING' }, inplace=True)
        df_pos_list_with_kpi.replace({'DATA_IN_FACING' : { 'both' : 'Ok', 'left_only' : 'No_FACING_data'}}, inplace=True)

    if 'PICOS' in kpi_list: 
        df_pos_list_with_kpi = df_pos_list_with_kpi.merge(df_picos_cut, on='SHIP_TO', how='left', indicator=True )
        #df_pos_list = df_pos_list.loc[df_pos_list['_merge'] != 'right_only'].reset_index().drop(columns='index')
        df_pos_list_with_kpi['_merge'] = df_pos_list_with_kpi['_merge'].astype('string')
        df_pos_list_with_kpi.rename(columns={'_merge': 'DATA_IN_PICOS' }, inplace=True)
        df_pos_list_with_kpi.replace({'DATA_IN_PICOS' : { 'both' : 'Ok', 'left_only' : 'No_PICOS_data'}}, inplace=True)

    df_pos_list_with_kpi['PAIR_NUMBER'] = 0
    df_pos_list_with_kpi['POS_TYPE'] = '-'
    return df_pos_list_with_kpi

In [ ]:
print("Кол-во контрольных ТТ:")
sl_number_control_pos = widgets.IntSlider(
    value=1,
    min=1,
    max=10,
    step=1,
    description='Кол-во ТТ:',
    disabled=False,
    continuous_update=True,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

sl_number_control_pos

In [ ]:
global df_pair_list
df_pair_list = pd.DataFrame([])


def find_pairs(test_pos_list, df_pos_list_with_kpi):

    global df_pair_list
    df_pair_list = pd.DataFrame([])
    
    
    
    for index, df_row in test_pos_list.iterrows():
        #Сохраняем данные по тестовой точке
        
        df_one_test_pos = df_pos_list_with_kpi.loc[df_pos_list_with_kpi['SHIP_TO'] ==  df_row['SHIP_TO']].reset_index()
        if df_one_test_pos.shape[0] == 0:
            df_one_test_pos.loc[0, ['SHIP_TO', 'RSD' ]] = [df_row['SHIP_TO'], 'POS not found in Optimum']
        df_one_test_pos['PAIR_NUMBER'] = index + 1
        df_one_test_pos['POS_TYPE'] = 'Test_POS'
        
        df_pair_list = pd.concat([df_pair_list, df_one_test_pos])
        df_pair_list = df_pair_list.reset_index().drop(columns=['index', 'level_0'])
        df_pair_list['SHIP_TO'] = df_pair_list['SHIP_TO'].astype('int')
        df_pair_list['PAIR_NUMBER'] = df_pair_list['PAIR_NUMBER'].astype('int')   
    
    
        ## Подбираем контрольную точку по территории
    
        df_used_pos = pd.DataFrame(pd.Series(pd.concat([pd.DataFrame(df_pair_list['SHIP_TO']), test_pos_list])['SHIP_TO'].unique(), name='SHIP_TO'))
        df_one_control_pos =  df_pos_list_with_kpi.merge(df_used_pos, how='left', on='SHIP_TO', indicator=True)
        df_one_control_pos =  df_one_control_pos.loc[df_one_control_pos['_merge'] == 'left_only'].drop(columns='_merge')
    
    
        if 'RSD' in feature_list_geo:
            df_one_control_pos = df_one_control_pos.loc[df_one_control_pos['RSD'] == df_one_test_pos['RSD'][0]  ]
        if 'BUSINESS_UNIT' in feature_list_geo:
            df_one_control_pos = df_one_control_pos.loc[df_one_control_pos['BUSINESS_UNIT'] == df_one_test_pos['BUSINESS_UNIT'][0]  ]
        if 'SALES_GROUP' in feature_list_geo:
            df_one_control_pos = df_one_control_pos.loc[df_one_control_pos['SALES_GROUP'] == df_one_test_pos['SALES_GROUP'][0]  ]
        if 'CITY' in feature_list_geo:
            df_one_control_pos = df_one_control_pos.loc[df_one_control_pos['CITY'] == df_one_test_pos['CITY'][0]  ]
        if 'CLIENT' in feature_list_geo:
            df_one_control_pos = df_one_control_pos.loc[df_one_control_pos['CLIENT'] == df_one_test_pos['CLIENT'][0]  ]
        if 'CUSTOMER_GROUP' in feature_list_geo:
            df_one_control_pos = df_one_control_pos.loc[df_one_control_pos['CUSTOMER_GROUP'] == df_one_test_pos['CUSTOMER_GROUP'][0]  ]
        
            ## Подбираем контрольную точку по KPI
        if 'OSA' in kpi_list: 
            df_one_control_pos = df_one_control_pos.loc[df_one_control_pos['OSA'] >= df_one_test_pos['OSA'][0]   *    kpi_ranges['OSA'][0] ]
            df_one_control_pos = df_one_control_pos.loc[df_one_control_pos['OSA'] <= df_one_test_pos['OSA'][0]   *    kpi_ranges['OSA'][1] ]
        if 'kCAF' in kpi_list: 
            df_one_control_pos = df_one_control_pos.loc[df_one_control_pos['kCAF'] >= df_one_test_pos['kCAF'][0]   *   kpi_ranges['kCAF'][0]  ]
            df_one_control_pos = df_one_control_pos.loc[df_one_control_pos['kCAF'] <= df_one_test_pos['kCAF'][0]   *   kpi_ranges['kCAF'][1]  ]
        if 'VISITS' in kpi_list: 
            df_one_control_pos = df_one_control_pos.loc[df_one_control_pos['PHOTO_AUDITS_PER_WEEK'] >= df_one_test_pos['PHOTO_AUDITS_PER_WEEK'][0]   *    kpi_ranges['VISITS'][0]  ]
            df_one_control_pos = df_one_control_pos.loc[df_one_control_pos['PHOTO_AUDITS_PER_WEEK'] <= df_one_test_pos['PHOTO_AUDITS_PER_WEEK'][0]   *    kpi_ranges['VISITS'][1]  ]
        if 'PICOS' in kpi_list: 
            df_one_control_pos = df_one_control_pos.loc[df_one_control_pos['PICOS_SCORE_FACT'] >= df_one_test_pos['PICOS_SCORE_FACT'][0]   *   kpi_ranges['PICOS'][0]  ]
            df_one_control_pos = df_one_control_pos.loc[df_one_control_pos['PICOS_SCORE_FACT'] <= df_one_test_pos['PICOS_SCORE_FACT'][0]   *   kpi_ranges['PICOS'][1]  ]
        if 'FACING' in kpi_list: 
            df_one_control_pos = df_one_control_pos.loc[df_one_control_pos['GROUP_FACT'] >= df_one_test_pos['GROUP_FACT'][0]   *   kpi_ranges['FACING'][0]  ]
            df_one_control_pos = df_one_control_pos.loc[df_one_control_pos['GROUP_FACT'] <= df_one_test_pos['GROUP_FACT'][0]   *   kpi_ranges['FACING'][1]  ]
        
        if df_one_control_pos.shape[0] > 0: 
            df_one_control_pos = df_one_control_pos.head(sl_number_control_pos.value)
            df_one_control_pos['PAIR_NUMBER'] = index + 1
            df_one_control_pos['POS_TYPE'] = 'Control_POS'
            df_pair_list = pd.concat([df_pair_list, df_one_control_pos])
        else: 
            df_pair_list.loc[(df_pair_list['SHIP_TO'] == df_row['SHIP_TO']) & (df_pair_list['POS_TYPE'] == 'Test_POS'), 'POS_TYPE' ] = 'Test_POS_wo_pair'
    return df_pair_list          

# ПОДБОР ПАР И ВЫГРУЗКА РЕЗУЛЬТАТОВ

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# Глобальная переменная, чтобы она существовала заранее


def on_calculate_click(button):
    
    output_calc.clear_output()
    
    with output_calc:
        try:
            df_sales_cut, df_osa_cut, df_visits_cut, df_facing_fact_cut, df_picos_cut = cut_dataframes(week_list, df_sales, df_osa, df_visits, df_facing_fact, df_picos )
            df_pos_list_with_kpi = calc_kpi_for_test_pos(df_pos_list, df_sales_cut, df_osa_cut, df_visits_cut, df_facing_fact_cut, df_picos_cut )
            df_pair_list = find_pairs(test_pos_list, df_pos_list_with_kpi)
            print("Все функции успешно выполнены")
            print('Найдено пар: ' + str(df_pair_list[df_pair_list['POS_TYPE'] == 'Test_POS'].shape[0]))
            print('Точки без пары: ' + str(df_pair_list[(df_pair_list['RSD'] != 'POS not found in Optimum') & (df_pair_list['POS_TYPE'] == 'Test_POS_wo_pair')].shape[0]))
            print('Точек нет в SFA: ' + str(df_pair_list[df_pair_list['RSD'] == 'POS not found in Optimum'].shape[0]))
            #display(df_pair_list)

        except Exception as e:
            print(f"Ошибка: {e}")
    return df_pair_list

calculate_button = widgets.Button(
    description='Рассчитать',
    layout=widgets.Layout(width='20%', height='40px'),
    button_style='danger',
    icon='check'
)


output_calc = widgets.Output()

calculate_button.on_click(on_calculate_click)

display(widgets.VBox([calculate_button, output_calc]))

---

In [ ]:
import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# Поле для ввода пути к файлу
path_widget = widgets.Text(
    value='./result.xlsx',
    placeholder='Например: C:/temp/df_pair_list.xlsx или ./df_pair_list.xlsx',
    description='Путь:',
    layout=widgets.Layout(width='700px')
)

# Чекбокс: сохранять ли индекс
index_widget = widgets.Checkbox(
    value=False,
    description='Сохранять индекс'
)

# Кнопка экспорта
export_button = widgets.Button(
    description='Выгрузить в Excel',
    layout=widgets.Layout(width='20%', height='40px'),
    button_style='success',
    icon='download'
)

# Поле для сообщений
output = widgets.Output()

def export_to_excel(button):
    with output:
        output.clear_output()
        
        try:
            file_path = path_widget.value.strip()
            
            if not file_path:
                print("Ошибка: укажите путь к файлу.")
                return
            
            if not file_path.lower().endswith('.xlsx'):
                file_path += '.xlsx'
            
            # Создаем папку, если она указана и не существует
            dir_name = os.path.dirname(file_path)
            if dir_name:
                os.makedirs(dir_name, exist_ok=True)
            
            # Проверка, что df_pair_list существует
            if 'df_pair_list' not in globals():
                print("Ошибка: DataFrame df_pair_list не найден.")
                return
            
            # Экспорт в Excel
            df_pair_list.to_excel(file_path, index=index_widget.value)
            
            print(f"Файл успешно сохранён: {os.path.abspath(file_path)}")
        
        except Exception as e:
            print(f"Ошибка при сохранении файла: {e}")

export_button.on_click(export_to_excel)

ui = widgets.VBox([
    path_widget,
    index_widget,
    export_button,
    output
])

display(ui)

In [ ]:

# Для того чтобы запускать приложение в один клик, надо создать ярлык на рабочем столе и вписать туда такою строку: 

# C:\Windows\System32\WindowsPowerShell\v1.0\powershell.exe -NoExit -Command "C:\Users\rokotyev\AppData\Local\miniconda3\python.exe -m voila 'C:\Users\rokotyev\Yandex.Disk\_Main Data Rokotyan\1. IT\test.ipynb' --port=8866"  

# ВАЖНО! Есть ограничение по длинне символов. Проверь что путь до тетрадки в него.